# Assessing Regulatory Compliance using Acquirium — Seawater RO Desalination

We should be able to write acquirium applications that use metadata and data to report regulatory compliance of plants.

In this direction, we picked two regulations that desalination plants has to comply with:

1. [**California Ocean Plan (2019), Chapter III.M.3**](https://www.waterboards.ca.gov/water_issues/programs/ocean/docs/oceanplan2019.pdf)— receiving-water limitation for the salinity of the brine discharge.
2. [**Cal. Code Regs. Tit. 22 § 64449**](https://www.law.cornell.edu/regulations/california/22-CCR-64449) — secondary MCLs (TDS) for the delivered drinking water.

Every check is written against ontology semantics (classes, media, substances, quantity kinds, boundaries), never against plant-specific point names.

In [ ]:
import polars as pl
from acquirium import Acquirium
pl.Config.set_tbl_width_chars(1000)
pl.Config.set_fmt_str_lengths(200)

acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)
S223 = "http://data.ashrae.org/standard223#"
NAWI = "urn:nawi-water-ontology#"

# The plant under audit. Compliance is assessed per plant, and this server also
# hosts the seawater-ro-fouled model, so the queries below are rooted at this
# system. Everything else stays semantic.
PLANT = "wbs:seawater-ro-plant"

## Ocean Plan, Chapter III.M.3 — Receiving Water Limitation for Salinity

> **III.M.3.b(1)** Discharges shall not exceed a **daily maximum of 2.0 parts per thousand (ppt) above natural background salinity** measured no further than 100 meters (328 ft) horizontally from each discharge point. There is no vertical limit to this zone.
>
> **III.M.3.b(2)** In determining an effluent limit necessary to meet this receiving water limitation, permit writers shall use the formula in chapter III.C.4 that has been modified for brine discharges as follows:
>
> $$C_e = C_o + D_m(2.0\,\text{ppt}) = (2.0\,\text{ppt} + C_s) + D_m(2.0\,\text{ppt})$$
>
> where 
>
>$C_e$ = effluent (discharge) concentration limit, 
>
>$C_o$ = salinity to be met at the completion of initial dilution, 
>
>$C_s$ = natural background salinity, 
>
>$D_m$ = minimum probable initial dilution (parts seawater per part brine).

This regulation concers the water quality at the intake and the discharge. 

The watertap desal RO flowsheet provides us the following quantities:
- effluent (discharge) salinity ($C_e$, the brine concentrate TDS) 
- natural background salinity ($C_s$, the intake seawater TDS) 

However, The initial-dilution factor $D_m$ is a property that lies **outside the model boundary**. It depends on the location of the plant and is determined based on a specific marine study over a long period of time. In this example we'll assume 237.0 taken from [this study](https://www.sejpa.org/images/Minimum_Initial_Dilution_Factor_Dm_Re-Evaluation_Study.pdf) 

First, we'll need to find the intake and discharge water salinity values. 

In [ ]:
# Both quantities live at the plant *boundary*: the ocean outfall is the boundary
# connection point that carries brine, and natural background salinity is what the
# plant draws from the ocean at its seawater boundary. Discovery is purely semantic
# below the plant root: no point names, just medium + substance + quantity kind.
def boundary_concentration_query(medium, alias):
    """Salt concentration observed at the plant's boundary point carrying `medium`."""
    return (acq.query().entity(uri=PLANT).alias("plant")
                .related("Connection Point").alias(f"{alias}_boundary")
                .where(medium=medium)
                .measurement(alias=alias).where(substance="constituent_salt",
                                                quantity_kind="mass concentration")
    )

discharge_q = boundary_concentration_query(NAWI + "Water-Brine", "discharge_salinity")
discharge_q.metadata()

In the discharge side, we have `"wbs:PXR-brine-out"` which is the Brine outlet of the pressure exchanger. It sits at the boundary of the entire plant and also the desalination subsystem

In [ ]:
background_q = boundary_concentration_query(NAWI + "Water-Seawater", "background_salinity")
background_q.metadata()

In the effluent side, we have `"wbs:intake-in"` which is the input of intake pump. It sits at the boundary of both the entire plant and the pretreatment subsystem.

In [ ]:
def get_daily_concentration(discharge_q, background_q):
    ce = discharge_q.dataframe(shape="wide", cast_value="float")
    cs = background_q.dataframe(shape="wide", cast_value="float")
    return (ce.join(cs, on="time")
            .with_columns(increment=pl.col("discharge_salinity") - pl.col("background_salinity")))
              
daily_concentration = get_daily_concentration(discharge_q, background_q)
daily_concentration 

In [ ]:
def ocean_plan_daily(daily_concentration, Dm, allowed_ppt=2.0):
    return (daily_concentration.group_by(pl.col("time").dt.date().alias("day"))
        .agg(samples=pl.len(),
                   Cs=pl.col("background_salinity").mean(),
                   Ce=pl.col("discharge_salinity").mean(),
                   plant_max_increment_ppt=pl.col("increment").max())
        .sort("day")
        .with_columns(allowed_increment_ppt=allowed_ppt * (1 + Dm),
                    Dm_required=pl.col("plant_max_increment_ppt") / allowed_ppt - 1)
        .with_columns(compliant_at_100m=pl.col("plant_max_increment_ppt") <= pl.col("allowed_increment_ppt"))
        .sort("day")
    )

# Equation 1 with the daily-maximum rule of III.M.3.b(1). 
Dm = 237.0  # minimum probable initial dilution, parts seawater per part brine

ocean_plan_daily(daily_concentration, Dm, allowed_ppt=2.0)


The brine leaves the plant at up to ~29 ppt above background — 14× the 2.0 ppt receiving-water limit, which is normal for RO concentrate and *not* a violation by itself. 

Consequently, the plant is compliant with this regulation

## Cal. Code Regs. Tit. 22 § 64449 — Secondary MCLs (Table 64449-B, TDS)

> The total dissolved solids of delivered drinking water shall not exceed the following contaminant levels: **Recommended 500 mg/L; Upper 1,000 mg/L; Short Term 1,500 mg/L.** Constituent concentrations ranging to the Upper level are acceptable only with State Board approval; to the Short Term level only temporarily pending construction of treatment facilities. Systems monitoring quarterly determine compliance on a **running annual average**.

This regulation cares about the delivered drinking water. This time we need to find the boundary sensor in the product effluent side.

In [ ]:
# Delivered-water TDS = salt concentration at the plant's potable (Fluid-Water) boundary.
product_query = boundary_concentration_query(S223 + "Fluid-Water", "product_tds")
product_query.metadata()

We found the tds sensor at the effluent boundary. Let's check it's unit:

In [ ]:
product_query.data().units()

It's in $KG/m^3$ , while the regulation is in $mg/L$. Acquirium can perform the unit conversion for us:

In [ ]:
df = product_query.data().convert_to("mg/l").dataframe(shape="wide")
df

In [ ]:
df.describe()

In [ ]:
def count_samples_over_limit(df):
    recommended_limit = 500  # mg/L
    upper_limit = 1000  # mg/L
    short_term_limit = 1500  # mg/L

    df = df.with_columns(
        recommended_exceedance=pl.when(pl.col("product_tds") > recommended_limit).then(1).otherwise(0),
        upper_exceedance=pl.when(pl.col("product_tds") > upper_limit).then(1).otherwise(0),
        short_term_exceedance=pl.when(pl.col("product_tds") > short_term_limit).then(1).otherwise(0)
    )

    summary = df.with_columns(
        total_samples=pl.count(),
        recommended_exceedances=pl.sum("recommended_exceedance"),
        upper_exceedances=pl.sum("upper_exceedance"),
        short_term_exceedances=pl.sum("short_term_exceedance")
    ).select([
        pl.col("total_samples").first().alias("Total Samples"),
        pl.col("recommended_exceedances").first().alias("Samples Over Recommended Limit (500 mg/L)"),
        pl.col("upper_exceedances").first().alias("Samples Over Upper Limit (1000 mg/L)"),
        pl.col("short_term_exceedances").first().alias("Samples Over Short-Term Limit (1500 mg/L)")
    ]).transpose(include_header=True)
    summary.columns = ["Metric", "Count"]
    return summary

count_samples_over_limit(df)



Delivered-water TDS runs at roughly 190–260 mg/L — every sample is below the 500 mg/L *recommended* level. Therefore the product water is compliant in terms of TDS levels